In [3]:
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# torch / torchvision 은 Colab 기본 버전을 그대로 씁니다. (건드리면 깨짐)
PACKAGES = [
    "transformers==4.55.2",
    "peft==0.17.0",
    "trl==0.21.0",
    "bitsandbytes==0.48.0",
    "accelerate==1.10.0",
    "datasets==4.0.0",
    "huggingface-hub<1.0",
]

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
    print("설치 완료")
else:
    print("로컬 환경이라 설치를 건너뜁니다.")

설치 완료


In [4]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import shutil
import torch
import pandas as pd
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftConfig, PeftModel

print("torch", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers", transformers.__version__)
if transformers.__version__ != "4.55.2":
    raise RuntimeError("라이브러리 버전이 다릅니다. Colab 메뉴 '런타임 > 세션 다시 시작'을 누른 뒤 이 셀부터 다시 실행하세요.")

# torch의 CUDA 버전과 맞는 파일이 없으면 여기서 바로 오류가 납니다.
import bitsandbytes
print("bitsandbytes", bitsandbytes.__version__)

torch 2.11.0+cu130 | CUDA 13.0
transformers 4.55.2
bitsandbytes 0.48.0


In [5]:
# 쓸 어댑터 하나만 주석을 푸세요. 데이터와 프롬프트는 이 값에 맞춰 자동으로 바뀝니다.
#ADAPTER_DIR = "qwen-python-answers"    # fine_tun_local.ipynb
ADAPTER_DIR = "qwen-glaive-python"   # fine_tun_glaive.ipynb
#ADAPTER_DIR = "qwen-koalpaca"        # fine_tun_koalpaca.ipynb

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. Colab 메뉴 '런타임 > 런타임 유형 변경'에서 T4 GPU를 고르세요.")
print(torch.cuda.get_device_name(0))

Tesla T4


In [6]:
# Colab에는 학습한 어댑터 폴더가 없으므로, 학습 노트북에서 내려받은 zip을 올려 풉니다.
if not os.path.isdir(ADAPTER_DIR):
    if not IN_COLAB:
        raise FileNotFoundError(f"{ADAPTER_DIR} 폴더가 없습니다. 먼저 학습 노트북을 실행하세요.")
    from google.colab import files
    print(f"{ADAPTER_DIR}.zip 파일을 고르세요.")
    uploaded = files.upload()
    shutil.unpack_archive(next(iter(uploaded)), ADAPTER_DIR)

# 어댑터가 어떤 베이스 모델로 학습됐는지는 adapter_config.json에 적혀 있습니다.
BASE_MODEL = PeftConfig.from_pretrained(ADAPTER_DIR).base_model_name_or_path
print(ADAPTER_DIR, "->", BASE_MODEL)

qwen-glaive-python.zip 파일을 고르세요.


Saving qwen-glaive-python.zip to qwen-glaive-python.zip
qwen-glaive-python -> Qwen/Qwen2.5-1.5B-Instruct


In [7]:
PYTHON_PREFIX = "Get answers for questions related to Python. Respond with answers, no explanation\n\n"

def prompt_for(question):
    # KoAlpaca는 지시 문구 없이 질문만 넣고 학습했습니다.
    if ADAPTER_DIR == "qwen-koalpaca":
        return [{"role": "user", "content": question}]
    return [{"role": "user", "content": PYTHON_PREFIX + question}]


In [8]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

def fits(item, max_length=256):
    # 템플릿을 문자열로 만든 뒤 토큰으로 나눕니다. transformers 버전에 따라 결과가 달라지지 않습니다.
    messages = prompt_for(item["question"]) + [{"role": "assistant", "content": item["answer"]}]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return len(tokenizer(text, add_special_tokens=False)["input_ids"]) <= max_length

def pick_val_set(rows, train_size=500, val_size=100, dedup=False):
    # 학습 노트북과 같은 규칙으로 고른 뒤, 학습에 쓴 앞쪽 500건 다음의 100건을 돌려줍니다.
    selected, seen = [], set()
    for item in rows:
        if dedup:
            question = item["question"].strip()
            if question in seen:
                continue
            seen.add(question)
        if fits(item):
            selected.append(item)
            if len(selected) == train_size + val_size:
                break
    return selected[train_size:]

In [9]:
if ADAPTER_DIR == "qwen-python-answers":
    if IN_COLAB and not os.path.exists("data/python_qa.csv"):
        # Colab에는 python_qa.csv가 없으므로 로컬의 data/python_qa.csv를 올립니다.
        from google.colab import files
        os.makedirs("data", exist_ok=True)
        print("python_qa.csv 파일을 고르세요.")
        uploaded = files.upload()
        shutil.move(next(iter(uploaded)), "data/python_qa.csv")
    qna_df = pd.read_csv("data/python_qa.csv")
    qa_format = [{"question": q, "answer": a} for q, a in zip(qna_df["Body"], qna_df["Answer"])]
    val_set = qa_format[3001:3101]

elif ADAPTER_DIR == "qwen-glaive-python":
    rows = load_dataset("glaiveai/glaive-code-assistant", split="train")
    rows = rows.filter(lambda item: "python" in item["question"].lower())
    val_set = pick_val_set(rows.shuffle(seed=42))

elif ADAPTER_DIR == "qwen-koalpaca":
    URL = re.compile(r"https?://")
    SENTENCE_END = re.compile(r"[.!?…)\]\"'다요죠음됨함]\s*$")
    rows = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
    rows = rows.filter(lambda item: not URL.search(item["output"].strip())
                       and bool(SENTENCE_END.search(item["output"].strip())))
    rows = rows.rename_columns({"instruction": "question", "output": "answer"})
    val_set = pick_val_set(rows.shuffle(seed=42), dedup=True)

else:
    raise ValueError(f"데이터를 모르는 어댑터입니다: {ADAPTER_DIR}")

len(val_set)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

c9bc9129-eba0-4b10-8292-4ae70fc7fa0d.jso(…):   0%|          | 0.00/219M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/136109 [00:00<?, ? examples/s]

Filter:   0%|          | 0/136109 [00:00<?, ? examples/s]

100

In [10]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [11]:
tokenizer.truncation_side = "left"   # 질문이 길면 앞을 자릅니다

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval();   # 드롭아웃을 끕니다. 세미콜론은 모델 구조 출력을 막습니다.

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [12]:
print("VRAM: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))

VRAM: 1.07 GB


In [13]:
def answer_for(question, max_new_tokens=200):
    text = tokenizer.apply_chat_template(prompt_for(question), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [14]:
item = val_set[0]
print(item["question"][:500])

How can I create a list in Python that stores the first 10 numbers divisible by 4?


In [15]:
print(answer_for(item["question"]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


To achieve this, you can use the following code:

```python
numbers = [i for i in range(10) if i % 4 == 0]
```

In this code, we utilize a list comprehension to generate the desired list. The `range(10)` function generates numbers from 0 to 9 (inclusive). We then apply an `if` condition using the modulo operator `%`, which checks whether each number is divisible by 4. If it is, the number is included in the resulting list.

After executing this code, the variable `numbers` will store the list `[0, 4, 8]`, representing the first three numbers divisible by 4 within the given range.


In [16]:
numbers = [i for i in range(10) if i % 4 == 0]

In [17]:
with model.disable_adapter():
    print(answer_for(item["question"]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


```python
numbers = [i for i in range(4, 41, 4)]
print(numbers)
```
This code creates a list of numbers from 4 to 40 (inclusive) that are divisible by 4 using a list comprehension and prints it.


In [18]:
numbers = [i for i in range(4, 41, 4)]
print(numbers)

[4, 8, 12, 16, 20, 24, 28, 32, 36, 40]


In [19]:
print(item["answer"])

You can use list comprehension in Python to create such a list. Here's an example code snippet that accomplishes this:

```python
divisible_by_four = [i for i in range(1,11) if i % 4 == 0]
```

In this code, we use the `range(1, 11)` function to generate numbers from 1 to 10 (inclusive). Then, we use the `%` operator to check if each number is divisible by 4. If the remainder of the division is 0, the number is considered divisible by 4, and it is included in the resulting list `divisible_by_four`.


In [20]:
val_set[2]

{'question': 'Is there a way to automate the process of copying a folder and all its files to a different location using Python?',
 'answer': "Yes, you can achieve this using the `shutil` module in Python. Here is an example code that demonstrates how to do it:\n\n```\nimport shutil\n\n# Define the source and destination paths\nsrc = '/path/to/source/folder/'\ndst = '/path/to/destination/folder/'\n\n# Copy the contents of the source folder to the destination folder\nshutil.copytree(src, dst)\n```\n\nThis code imports the `shutil` module, which provides various file operations. By specifying the source (`src`) and destination (`dst`) paths, you can use the `copytree` function to recursively copy the entire contents of the source folder to the destination folder."}

In [21]:
print(answer_for(val_set[2]['question']))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Yes, you can achieve this by utilizing the `shutil` module in Python. Here's an example code snippet that demonstrates how to copy a folder and its contents:

```python
import shutil

src = '/path/to/source/folder'
dst = '/path/to/destination/folder'

shutil.copytree(src, dst)
```

In this code, replace `/path/to/source/folder` with the actual path of the source folder you want to copy, and `/path/to/destination/folder` with the desired destination folder.

The `shutil.copytree()` function is used here instead of `shutil.copy()` because it allows us to recursively copy folders and their contents. It creates a new directory structure at the specified destination and copies all the files from the source folder into the destination folder.

By executing this code, your source folder will be copied to the designated destination folder, preserving all its files and subfolders.


In [22]:
print(answer_for("How do I remove duplicates from a list while keeping the order?"))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


To remove duplicates from a list in Python and maintain the original order of elements, you can use the following code:

```python
def remove_duplicates(lst):
    seen = set()
    return [x for x in lst if not (x in seen or seen.add(x))]

lst = [1, 2, 3, 4, 5]
print(remove_duplicates(lst))
```

In this solution, we define a function called `remove_duplicates` that takes an input list as its parameter. Inside the function, we create an empty set called `seen`. Then, we use a list comprehension to iterate over each element in the input list (`lst`). For each element, we check if it is already present in the `seen` set using the `in` keyword. If it is not found in the set, we add it to the set using the `add()` method.

By doing so, we ensure that only unique elements remain in the output list. Finally, we call


In [23]:
print(answer_for("네이버는 무엇을 하는 회사인가요?"))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


네이버는 한국의 인터넷 서비스 기업으로, 네트워크를 통해 제공하는 다양한 서비스와 제품들을 개발하고 판매합니다. 네이버는 주로 웹사이트, 소프트웨어, 뉴스, 검색 엔진, 게임, 앱 등 다양한 분야에서 활동하고 있습니다.

네이버의 주요 서비스들 중 일부는 다음과 같습니다:

1. 네이버 웹사이트: 네이버의 대표적인 웹사이트로, 정보 검색, 게시판, 카페, 뉴스, 영화 리뷰, 음악 리뷰 등을 제공합니다.
2. 네이버 검색 엔진: 네이버 웹사이트의 검색 기능으로, 사용자가 원하는 정보를 쉽게 찾고 찾아볼 수 있도록 도와줍니다.
3. 네이버 블로그:


In [24]:
import gradio as gr

In [25]:
def compare(question):
    tuned = answer_for(question)
    with model.disable_adapter():
        base = answer_for(question)
    return tuned, base

In [26]:
UI_BY_ADAPTER = {
    "qwen-python-answers": {
        "title": "Python Q&A 비교: Stack Overflow 파인튜닝",
        "description": "python_qa.csv(2008~2016년 Stack Overflow 질문과 답변)로 학습한 어댑터입니다. 영어로 질문하세요.",
        "label": "Python 질문 (영어)",
        "examples": [
            "What are the best books to learn Python?",
            "How do I read a file line by line in Python?",
        ],
    },
    "qwen-glaive-python": {
        "title": "Python Q&A 비교: glaive-code-assistant 파인튜닝",
        "description": "glaive-code-assistant의 Python 질문과 답변으로 학습한 어댑터입니다. 영어로 질문하세요.",
        "label": "Python 질문 (영어)",
        "examples": [
            "How can I sort a list of dictionaries by a value in Python?",
            "What is the difference between a list and a tuple in Python?",
        ],
    },
    "qwen-koalpaca": {
        "title": "한국어 Q&A 비교: KoAlpaca 파인튜닝",
        "description": "KoAlpaca-v1.1a(네이버 지식iN 질문과 ChatGPT 답변)로 학습한 어댑터입니다. 한국어로 질문하세요. 답의 사실 여부는 따로 확인하세요.",
        "label": "질문 (한국어)",
        "examples": [
            "겨울철 실내 습도를 적당히 유지하는 방법은 무엇인가요?",
            "바다는 왜 파란색으로 보이나요?",
        ],
    },
}
UI = UI_BY_ADAPTER[ADAPTER_DIR]

In [27]:
demo = gr.Interface(
    fn=compare,
    inputs=gr.Textbox(label=UI["label"], lines=5),
    outputs=[
        gr.Textbox(label="파인튜닝한 Qwen"),
        gr.Textbox(label="원래 Qwen"),
    ],
    title=UI["title"],
    description=UI["description"],
    examples=UI["examples"],
    flagging_mode="never",
)
demo.launch(share=IN_COLAB)   # Colab에서는 공개 링크가 있어야 제출이 동작합니다

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a89907f32442954817.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [28]:
demo.launch(share=IN_COLAB, auth=("myid", "mypassword"))

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a89907f32442954817.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [29]:
demo.close()

Closing server running on port: 7860
